# OMIn GraphRAG : 4-Triplet Extraction

**Goal:** Extract structured triplets `(head_entity, relation, tail_entity, record_id)` from every record using Gemini.
The KEO paper (https://arxiv.org/pdf/2510.05524) defines 14 relation types borrowed from REBEL/Wikidata plus domain-specific ones: OWNED BY, INSTANCE OF, FOLLOWED BY, HAS CAUSE, FOLLOWS, EVENT DISTANCE, HAS EFFECT, LOCATION, USED BY, INFLUENCED BY, TIME PERIOD, PART OF, MAINTAINED BY, DESIGNED BY. 
The problem they acknowledge: each RE tool under evaluation employed a distinct relation schema, making a unified global gold standard impractical. The loose count was introduced to mitigate limitations & only requiring that the LLM extract a plausible relation type from the correct record ID.
Looking at their distribution table, HAS CAUSE and HAS EFFECT dominate heavily (93 and 94 loose counts) while FOLLOWED BY, EVENT DISTANCE, INFLUENCED BY, DESIGNED BY have near-zero occurrences, meaning those relation types are essentially noise in this domain.
Our fix: we design a tighter, aviation-maintenance-specific relation schema, dropping the useless ones and adding semantically precise ones.
**Relation schema (10 types):**
- `CAUSES` component/event causes another event/failure ==> (replaces HAS CAUSE — directional, cleaner)
- `RESULTS_IN` action/event results in outcome ==> (replaces HAS EFFECT)
- `PART_OF` component is part of a larger system ==> (keeping it, well-defined)
- `LOCATED_IN` entity is located in a place/system ==> (replaces LOCATION, noun form)
- `REQUIRES_ACTION` problem requires a maintenance action ==> (new : for problem→action pairs)
- `INSPECTED_BY` component inspected by role/tool ==> (new :maintenance specific)
- `REPLACED_BY` component replaced by another ==> (new : very common in MaintNet data)
- `PRECEDED_BY` event preceded by another event ==> (replaces FOLLOWED BY/FOLLOWS ==>less ambiguous)
- `AFFECTS` entity affects another entity/system ==> (new : component affects system)
- `INSTANCE_OF` entity is an instance of a type/class
We drop: OWNED BY, EVENT DISTANCE, INFLUENCED BY, DESIGNED BY, TIME PERIOD, USED BY — either too rare, too vague, or not meaningful for our retrieval use case.

**Summary:** 
Used Gemini to extract 40,604 triplets (head, relation, tail, record_id) from those records. Every triplet carries the record_id of the record it came from as well as the row number for more precision. Then embedded all 15,487 unique entities with gemini-embedding-001.







## Setup & installations

In [ ]:
import sys
sys.stdout.flush()

In [ ]:
!pip install -q google-genai tqdm pandas

In [ ]:
import json, os, time, re
import pandas as pd
from tqdm.notebook import tqdm
from google import genai

GEMINI_API_KEY   = ""
client           = genai.Client(api_key=GEMINI_API_KEY)
EXTRACTION_MODEL = "gemini-2.5-flash-lite"
EMBEDDING_MODEL  = "gemini-embedding-001"

os.makedirs("./output", exist_ok=True)

with open("./output/records.json") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
print(f"Sources: {set(r['source'] for r in records)}")

## defining relation schema and prompt

In [ ]:
RELATION_TYPES = [
    "CAUSES", "RESULTS_IN", "PART_OF", "LOCATED_IN",
    "REQUIRES_ACTION", "INSPECTED_BY", "REPLACED_BY",
    "PRECEDED_BY", "AFFECTS", "INSTANCE_OF"
]

DOMAIN_TERMS = [
    "engine", "cylinder", "gasket", "hydraulic", "magnetos", "fuel pump",
    "landing gear", "aileron", "throttle", "carburetor", "brake", "propeller",
    "fuel tank", "ignition", "compressor", "alternator", "battery", "rotor",
    "fuselage", "elevator", "rudder", "flap", "avionics", "pitot tube"
]

EXTRACTION_PROMPT = '''You are an expert aviation maintenance knowledge engineer.
Your task is to extract knowledge graph triplets from aviation maintenance records.

RELATION TYPES (use ONLY these, exactly as written):
- CAUSES: a component, condition, or event directly causes a failure or event
- RESULTS_IN: an action or event leads to an outcome or consequence
- PART_OF: a component belongs to or is a subsystem of a larger system
- LOCATED_IN: an entity exists within a physical location or system
- REQUIRES_ACTION: a problem or condition requires a specific maintenance action
- INSPECTED_BY: a component or system is inspected by a person, role, or tool
- REPLACED_BY: a faulty component is replaced by a new or different component
- PRECEDED_BY: an event or condition was preceded by another event or condition
- AFFECTS: an entity, condition, or failure affects another entity or system
- INSTANCE_OF: an entity is a specific instance or example of a broader category

AVIATION DOMAIN TERMS for reference: {domain_terms}

RULES:
1. Expand all abbreviations (ACFT->aircraft, PLT->pilot, ENG->engine, TKOF->takeoff,
   HYD->hydraulic, GR->gear, FLT->flight, INSP->inspection, REPL->replaced, LDG->landing,
   CYL->cylinder, CARB->carburetor, BATT->battery, COMP->compressor, ALT->alternator)
2. Normalize entities to lowercase noun phrases (e.g. "fuel pump" not "FUEL PUMP")
3. Be specific: "hydraulic brake" not just "brake"; "left main gear" not just "gear"
4. Only extract triplets clearly supported by the text - no inference beyond what is stated
5. For PROBLEM/ACTION records: the problem REQUIRES_ACTION the action
6. Output ONLY triplets, one per line: <head_entity | RELATION | tail_entity>
7. If no clear triplets can be extracted, output: <NONE>
8. Maximum 6 triplets per record - prefer quality over quantity

EXAMPLES:
Text: "ENGINE IDLE OVERRIDE KILLED ENGINE. REMOVED & REPLACED FUEL SERVO."
<engine idle override | CAUSES | engine failure>
<engine failure | REQUIRES_ACTION | fuel servo replacement>
<fuel servo | REPLACED_BY | new fuel servo>
<fuel servo | PART_OF | engine fuel system>

Text: "TAILWHEEL COCKED RIGHT PRIOR TO TAKEOFF. PILOT ABORTED TAKEOFF."
<tailwheel misalignment | CAUSES | takeoff abort>
<tailwheel | PART_OF | landing gear>
<tailwheel misalignment | PRECEDED_BY | taxi operation>

Text: "HYDRAULIC LEAK IN MAIN GEAR. INSPECTION FOUND CRACKED SEAL. SEAL REPLACED."
<hydraulic leak | LOCATED_IN | main landing gear>
<cracked seal | CAUSES | hydraulic leak>
<cracked seal | REQUIRES_ACTION | seal replacement>
<seal | REPLACED_BY | new seal>
<hydraulic system | AFFECTS | landing gear operation>

Now extract triplets from this text:
Text: "{text}"
Triplets:'''

print("Prompt ready. Relations:", RELATION_TYPES)

## Extraction function + test 
before running the whole extraction process, since it takes too long) we test on a sample of 5 records, mixed from FAA and annotations 
before running next cell to launch extraction we should make sure that in the test : 
-entities are lowercase and specific
-relations are from our 10-type schema only
-abbreviations are expanded (TKOF→takeoff, ENG→engine, PLT→pilot)
-triplets are grounded in the source text

In [ ]:
def extract_triplets(text, record_id, client, model=EXTRACTION_MODEL):
    prompt = EXTRACTION_PROMPT.format(
        domain_terms=", ".join(DOMAIN_TERMS),
        text=text[:1500]
    )
    for attempt in range(4):  # up to 4 attempts
        try:
            response = client.models.generate_content(
                model=model, contents=prompt,
                config={"temperature": 0.1}
            )
            raw = response.text.strip()
            break  # success
        except Exception as e:
            err = str(e)
            if "503" in err or "UNAVAILABLE" in err:
                wait = 30 * (attempt + 1)  # 30s, 60s, 90s, 120s
                tqdm.write(f"  503 overload for {record_id}, waiting {wait}s (attempt {attempt+1}/4)")
                time.sleep(wait)
            elif "429" in err or "RATE" in err:
                wait = 10 * (attempt + 1)
                tqdm.write(f"  429 rate limit for {record_id}, waiting {wait}s")
                time.sleep(wait)
            else:
                tqdm.write(f"  API error for {record_id}: {e}")
                return None
            if attempt == 3:
                return None
            continue
    else:
        return None

    triplets = []
    for line in raw.split("\n"):
        line = line.strip()
        if not line or line == "<NONE>":
            continue
        match = re.search(r'<?([^|<>]+)\|\s*([A-Z_]+)\s*\|([^|<>]+)>?', line)
        if match:
            head     = match.group(1).strip().lower()
            relation = match.group(2).strip().upper()
            tail     = match.group(3).strip().lower()
            if relation in RELATION_TYPES and len(head) > 1 and len(tail) > 1:
                triplets.append({
                    "head": head, "relation": relation, "tail": tail,
                    "record_id": record_id, "source_text": text[:200]
                })
    return triplets


# Test: 2 FAA + 3 annotation records
test_records = records[:2] + records[2750:2753]
print("=== EXTRACTION TEST (5 records) ===\n")
for rec in test_records:
    print(f"[{rec['record_id']}] {rec['text'][:150]}")
    triplets = extract_triplets(rec['text'], rec['record_id'], client)
    if triplets:
        for t in triplets:
            print(f"  → {t['head']} | {t['relation']} | {t['tail']}")
    else:
        print("  → No triplets extracted")
    print()
#test results 
# Abbreviations expanded correctly (TKOF not expanded but that's fine since it's in the entity, not the relation)
# Relations are clean and from our 10-type schema only
# Entities are specific ("cylinder 2 lower spark plug", not just "plug")
# Causal chains preserved (FAA_1: settling → perceived trouble → release → tree impact)
# PROBLEM/ACTION pairs correctly mapped to REQUIRES_ACTION + RESULTS_IN
# PART_OF hierarchy being built (spark plug → cylinder 2 → engine)

## full extraction — all 8917 records 
we also added checkpointing in case there is some interruption so we don't lose progress.

In [ ]:
!pip install aiohttp

In [ ]:
import asyncio
import aiohttp
import json, os, time, re
from tqdm.notebook import tqdm

CHECKPOINT_FILE = "./output/triplets_checkpoint.json"
FINAL_FILE      = "./output/triplets.json"
BATCH_SIZE      = 100
MAX_CONCURRENT  = 10  # 10 parallel requests at once
MAX_RETRIES     = 4

# resume from checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    all_triplets  = ckpt["triplets"]
    processed_ids = set(ckpt["processed_ids"])
    print(f"Resuming: {len(processed_ids)} done, {len(all_triplets)} triplets so far")
else:
    all_triplets  = []
    processed_ids = set()

remaining   = [r for r in records if r["record_id"] not in processed_ids]
empty_count = 0
print(f"Records remaining: {len(remaining)}")

async def extract_triplets_async(session, text, record_id, api_key, model):
    prompt = EXTRACTION_PROMPT.format(
        domain_terms=", ".join(DOMAIN_TERMS),
        text=text[:1500]
    )
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.1}
    }
    for attempt in range(MAX_RETRIES):
        try:
            async with session.post(url, json=payload, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                if resp.status == 503:
                    wait = 30 * (attempt + 1) #adding this because we had high demand on the model we chose, so that it retries 4 times and waits 30 seconds each time so that it doesn't just crash directly
                    print(f"  503 {record_id}, waiting {wait}s")
                    await asyncio.sleep(wait)
                    continue
                if resp.status == 429:
                    wait = 10 * (attempt + 1)
                    print(f"  429 {record_id}, waiting {wait}s")
                    await asyncio.sleep(wait)
                    continue
                if resp.status != 200:
                    print(f"  Error {resp.status} for {record_id}")
                    return record_id, []
                data = await resp.json()
                raw  = data["candidates"][0]["content"]["parts"][0]["text"].strip()
        except Exception as e:
            print(f"  Exception {record_id}: {e}")
            await asyncio.sleep(10)
            continue

        triplets = []
        for line in raw.split("\n"):
            line = line.strip()
            if not line or line == "<NONE>":
                continue
            match = re.search(r'<?([^|<>]+)\|\s*([A-Z_]+)\s*\|([^|<>]+)>?', line)
            if match:
                head     = match.group(1).strip().lower()
                relation = match.group(2).strip().upper()
                tail     = match.group(3).strip().lower()
                if relation in RELATION_TYPES and len(head) > 1 and len(tail) > 1:
                    triplets.append({
                        "head": head, "relation": relation, "tail": tail,
                        "record_id": record_id, "source_text": text[:200]
                    })
        return record_id, triplets
    return record_id, []  # all retries failed
async def run_extraction(records_to_process, api_key, model):
    global all_triplets, processed_ids, empty_count

    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def bounded_extract(session, rec):
        async with semaphore:
            if len(rec["text"].strip()) < 15:
                return rec["record_id"], []
            return await extract_triplets_async(
                session, rec["text"], rec["record_id"], api_key, model
            )

    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_extract(session, rec) for rec in records_to_process]
        pbar  = tqdm(total=len(tasks), desc="Extracting")

        for i, coro in enumerate(asyncio.as_completed(tasks)):
            record_id, triplets = await coro
            pbar.update(1)

            if not triplets:
                empty_count += 1
            else:
                all_triplets.extend(triplets)

            processed_ids.add(record_id)

            #checkpoint every BATCH_SIZE records
            if (i + 1) % BATCH_SIZE == 0:
                with open(CHECKPOINT_FILE, "w") as f:
                    json.dump({"triplets": all_triplets,
                               "processed_ids": list(processed_ids)}, f)
                pbar.write(f"Checkpoint: {len(processed_ids)} records, {len(all_triplets)} triplets")

        pbar.close()

await run_extraction(remaining, GEMINI_API_KEY, "gemini-2.5-flash-lite")

# Save final
with open(FINAL_FILE, "w") as f:
    json.dump(all_triplets, f, indent=2)

print(f"\n=== DONE ===")
print(f"Total records processed:  {len(processed_ids)}")
print(f"Empty records:            {empty_count}")
print(f"Total triplets:           {len(all_triplets)}")

## inspecting triplet extracion, 
added because there were crashes and some records weren't properly processed, so some triplets were missing. 


In [ ]:
#records that are marked processed but have zero triplets
import json

with open("./output/triplets_checkpoint.json") as f:
    ckpt = json.load(f)

with open("./output/records.json") as f:
    records = json.load(f)
#set of record_ids that actually have triplets
records_with_triplets = set(t["record_id"] for t in ckpt["triplets"])
processed_ids         = set(ckpt["processed_ids"])
#records marked done but with no triplets
missing = []
for r in records:
    rid = r["record_id"]
    if rid in processed_ids and rid not in records_with_triplets:
        #might be be legitimately empty or a failed extraction
        # checking if text is long enough to expect triplets
        if len(r["text"].strip()) >= 15:
            missing.append(r)

print(f"Records marked processed but no triplets: {len(missing)}")
print(f"Sample missing record IDs: {[r['record_id'] for r in missing[:10]]}")

In [ ]:
# Reprocess the 305 missing records
print(f"Reprocessing {len(missing)} records...")

async def repair_run(missing_records, api_key, model):
    semaphore = asyncio.Semaphore(5)  # lower concurrency for reliability
    repair_triplets = []

    async def bounded_extract(session, rec):
        async with semaphore:
            return await extract_triplets_async(
                session, rec["text"], rec["record_id"], api_key, model
            )

    connector = aiohttp.TCPConnector(limit=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_extract(session, rec) for rec in missing_records]
        pbar  = tqdm(total=len(tasks), desc="Repairing")

        for coro in asyncio.as_completed(tasks):
            record_id, triplets = await coro
            pbar.update(1)
            if triplets:
                repair_triplets.append((record_id, triplets))
                pbar.write(f"  ✓ {record_id}: {len(triplets)} triplets")
            else:
                pbar.write(f"  ✗ {record_id}: still empty")
        pbar.close()

    return repair_triplets

repair_results = await repair_run(missing, GEMINI_API_KEY, "gemini-2.5-flash-lite")

# Merge repair triplets into main triplets file
with open("./output/triplets.json") as f:
    all_triplets = json.load(f)

before = len(all_triplets)
for record_id, triplets in repair_results:
    all_triplets.extend(triplets)

with open("./output/triplets.json", "w") as f:
    json.dump(all_triplets, f, indent=2)

print(f"\nRepair complete:")
print(f"  Records reprocessed: {len(missing)}")
print(f"  Records recovered:   {len(repair_results)}")
print(f"  New triplets added:  {len(all_triplets) - before}")
print(f"  Total triplets now:  {len(all_triplets)}")

## Triplet quality analysis

In [ ]:
with open(FINAL_FILE) as f:
    all_triplets = json.load(f)

df = pd.DataFrame(all_triplets)

print("**** Triplets stats ****")
print(f"Total triplets:    {len(df)}")
print(f"Unique heads:      {df['head'].nunique()}")
print(f"Unique tails:      {df['tail'].nunique()}")
print(f"Unique record_ids: {df['record_id'].nunique()}")

print("\n**** Relations distribution ****")
print(df['relation'].value_counts().to_string())

print("\n**** Top 20 head entities ****")
print(df['head'].value_counts().head(20).to_string())

print("\n**** Sample triplets per relation****")
for rel in RELATION_TYPES:
    sample = df[df['relation'] == rel].head(2)
    if len(sample) > 0:
        print(f"\n{rel}:")
        for _, row in sample.iterrows():
            print(f"  {row['head']} | {rel} | {row['tail']}  [{row['record_id']}]")

## Embedding all unique entities with Gemini gemini-embedding-001

In [ ]:
EMBEDDINGS_FILE  = "./output/entity_embeddings.json"
EMBEDDING_MODEL  = "gemini-embedding-001"  # current stable embedding model

all_entities = sorted(set(df['head'].tolist() + df['tail'].tolist()))
print(f"Unique entities to embed: {len(all_entities)}")

if os.path.exists(EMBEDDINGS_FILE):
    with open(EMBEDDINGS_FILE) as f:
        entity_embeddings = json.load(f)
    print(f"Loaded {len(entity_embeddings)} cached embeddings")
else:
    entity_embeddings = {}

to_embed = [e for e in all_entities if e not in entity_embeddings]
print(f"Need to embed: {len(to_embed)}")

EMBED_BATCH = 50  # smaller batch = more stable

for i in tqdm(range(0, len(to_embed), EMBED_BATCH), desc="Embedding"):
    batch = to_embed[i:i+EMBED_BATCH]
    success = False
    for attempt in range(3):
        try:
            result = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch
            )
            for entity, emb in zip(batch, result.embeddings):
                entity_embeddings[entity] = emb.values
            success = True
            break
        except Exception as e:
            print(f"  Error batch {i} attempt {attempt+1}: {e}")
            time.sleep(10 * (attempt + 1))
    
    if not success:
        print(f"  Skipping batch {i} after 3 failed attempts")
    
    time.sleep(0.5)
    
    # Save every 10 batches
    if (i // EMBED_BATCH) % 10 == 0:
        with open(EMBEDDINGS_FILE, "w") as f:
            json.dump(entity_embeddings, f)
        tqdm.write(f"  Saved {len(entity_embeddings)} embeddings so far")

with open(EMBEDDINGS_FILE, "w") as f:
    json.dump(entity_embeddings, f)

print(f"Saved {len(entity_embeddings)} entity embeddings")
print(f"Embedding dimension: {len(list(entity_embeddings.values())[0])}")

## Embedding relation types (used in guardrail schema comparison)

In [ ]:
RELATION_DESCRIPTIONS = {
    "CAUSES":          "a component condition or event directly causes a failure or event",
    "RESULTS_IN":      "an action or event leads to an outcome or consequence",
    "PART_OF":         "a component belongs to or is a subsystem of a larger system",
    "LOCATED_IN":      "an entity exists within a physical location or system",
    "REQUIRES_ACTION": "a problem or condition requires a specific maintenance action",
    "INSPECTED_BY":    "a component or system is inspected by a person role or tool",
    "REPLACED_BY":     "a faulty component is replaced by a new or different component",
    "PRECEDED_BY":     "an event or condition was preceded by another event or condition",
    "AFFECTS":         "an entity condition or failure affects another entity or system",
    "INSTANCE_OF":     "an entity is a specific instance or example of a broader category"
}

rel_texts = [RELATION_DESCRIPTIONS[r] for r in RELATION_TYPES]
result = client.models.embed_content(model=EMBEDDING_MODEL, contents=rel_texts)
relation_embeddings = {rel: emb.values for rel, emb in zip(RELATION_TYPES, result.embeddings)}

with open("./output/relation_embeddings.json", "w") as f:
    json.dump(relation_embeddings, f)

print(f"Saved {len(relation_embeddings)} relation embeddings")

In [ ]:
import json

with open("./output/records.json") as f:
    records = json.load(f)

#adding source_row field to every record ==> for citation precision
for rec in records:
    rid = rec["record_id"]
    if rid.startswith("FAA_"):
        rec["source_row"] = int(rid.replace("FAA_", ""))
    elif rid.startswith("ANN_"):
        rec["source_row"] = rec["metadata"].get("ident", rid.replace("ANN_", ""))

with open("./output/records.json", "w") as f:
    json.dump(records, f, indent=2)

print("Added source_row to all records")
print(json.dumps(records[0], indent=2))
print(json.dumps(records[2750], indent=2))